# Sionna RT export

> Convert any MultiGrid environment layout into a Sionna RT scene (and optionally a Blender scene).

In [ ]:
#| default_exp utils.sionna_export

In [ ]:
#| hide
from nbdev.showdoc import *

## Overview

```text
Every grid cell is mapped to a category (outer wall, inner wall, door, key, ball, box, ...),
and each category to one of Sionna RT's ITU radio materials (ITU-R P.2040, see
https://nvlabs.github.io/sionna/rt/api/radio_materials.html).

Outputs (in --out; default sionna_scenes/<env>_seed<seed>[_<kwargs hash>], so different
seeds / settings never share a folder, and an existing scene is only replaced with --overwrite):
    scene.xml            Sionna RT scene (Mitsuba 3 XML), load with `sionna.rt.load_scene`
    meshes/*.ply         one mesh per category (binary PLY, z-up, metres)
    scene.json           metadata: grid -> world mapping, materials, agents / goals in world coordinates
    scene.blend          (with --blend) the same scene in Blender, materials named `itu_<type>`
                         (or later, for an exported folder: --build-blend <folder>)

Opening in Blender
    Open scene.blend (File > Open). Build it with the Blender you will open it with
    (--blender /path/to/blender, or the BLENDER environment variable): a .blend saved by a
    newer Blender (e.g. the 5.x `bpy` module) cannot be opened by an older one (e.g. 4.2).
    Importing scene.xml with the mitsuba-blender add-on is not recommended: its importer needs the `mitsuba` package inside Blender (and an older Mitsuba
    version, it uses `mitsuba.xml_to_props`), and it skips Sionna's `itu-radio-material` materials.

Geometry
    * walls and closed / locked doors are extruded cells; only their outer surfaces are kept
      (faces between neighbouring solid cells are dropped), so rays never hit internal faces
    * open doors are left as openings; keys, balls and boxes are small objects (optional)
    * a ground plane (and optionally a ceiling) covers the grid

Coordinates (verified by ray casting every cell in Sionna RT)
    MultiGrid uses image coordinates: x to the right, y DOWN, headings right / down / left / up =
    (+1,0) / (0,+1) / (-1,0) / (0,-1). Mitsuba / Sionna RT are right-handed with z UP, in metres.
        world = origin + cell_size * (x + 0.5, -(y + 0.5), 0)       (cell centre)
    i.e. grid right -> world +x (east), grid up -> world +y (north), so the top view is not mirrored.
    By default the grid is centred at the world origin (--no-center: top-left corner at the origin).
    Agent headings map to world +x / -y / -x / +y; scene.json gives each agent's `world_heading`
    and `yaw` (radians, counter-clockwise from +x), e.g. `Transmitter(..., orientation=[yaw, 0, 0])`.

    The meshes are written directly in this z-up frame. When (re-)exporting from Blender with the
    mitsuba-blender add-on, set Forward: Y, Up: Z: the add-on's defaults (Forward -Z, Up Y) convert
    to Mitsuba's y-up convention, and the scene then lies on its side in Sionna.

Materials
    Each material is written as an explicit `itu-radio-material` with its thickness, and named
    `mat-itu_<type>` (Sionna's naming convention), so the scene loads without any extra code.
    In the .blend file, materials are named `itu_<type>`: re-exporting it with the mitsuba-blender
    add-on gives `mat-itu_<type>` ids, which Sionna maps to the same ITU materials (with Sionna's
    default thickness, since the add-on does not export it).

    Sionna applies a material's thickness at every surface a ray crosses. A ray through one wall
    cell crosses two faces, so the default wall thickness is cell_size / 2 per face.

Examples
    python -m multigrid.utils.sionna_export --env MultiGrid-FindGoal-15x15-v0 --seed 0 \
        --env-kwargs num_obstacles=6 agents=2 --out scenes/findgoal_s0 --blend --frequency 3.5e9
    python -m multigrid.utils.sionna_export --env MultiGrid-RedBlueDoors-8x8-v0 --agents 2 \
        --material inner_wall=metal --ceiling ceiling_board --out scenes/rbd

    From code, with any env you created / reset yourself:
        from multigrid.utils.sionna_export import SceneConfig, export_scene
        export_scene(env, 'scenes/my_env', SceneConfig(cell_size=1.0), blend=True)

    In Sionna RT:
        from sionna.rt import load_scene
        scene = load_scene('scenes/findgoal_s0/scene.xml')
```

In [ ]:
#| export
from __future__ import annotations
import argparse
import contextlib
import hashlib
import io
import json
import math
import os
import subprocess
import sys
from dataclasses import asdict, dataclass, field
import numpy as np

In [ ]:
#| export
# ITU materials supported by Sionna RT (sionna.rt.radio_materials.itu), with their valid
# frequency ranges in GHz (a material can only be evaluated inside one of its ranges)
ITU_MATERIALS = {
    'vacuum': [(0.001, 100)],
    'concrete': [(1, 100), (110, 330)],
    'brick': [(1, 40), (110, 330)],
    'plasterboard': [(1, 100), (100, 400)],
    'wood': [(0.001, 100), (100, 400)],
    'glass': [(0.1, 100), (100, 450)],
    'clear_acrylic': [(110, 330)],
    'ceiling_board': [(1, 100), (100, 450)],
    'chipboard': [(1, 100), (100, 200)],
    'plywood': [(1, 40), (100, 400)],
    'marble': [(1, 60), (100, 400)],
    'floorboard': [(50, 100), (100, 450)],
    'vinyl_tile': [(1, 40)],
    'carpet_tile': [(1, 40)],
    'asphalt_concrete': [(1, 40)],
    'metal': [(1, 100)],
    'very_dry_ground': [(1, 10)],
    'medium_dry_ground': [(1, 10)],
    'wet_ground': [(1, 10)],
}

In [ ]:
#| export
# Display colours (only used for rendering / Blender viewport)
ITU_COLORS = {
    'concrete': (0.54, 0.54, 0.54), 'brick': (0.70, 0.32, 0.22), 'wood': (0.55, 0.38, 0.20),
    'metal': (0.75, 0.77, 0.80), 'glass': (0.60, 0.80, 0.90), 'plasterboard': (0.90, 0.90, 0.86),
    'ceiling_board': (0.95, 0.95, 0.95), 'chipboard': (0.72, 0.58, 0.40), 'plywood': (0.80, 0.65, 0.45),
    'marble': (0.92, 0.90, 0.88), 'floorboard': (0.60, 0.45, 0.30), 'vinyl_tile': (0.70, 0.70, 0.65),
    'carpet_tile': (0.35, 0.35, 0.45), 'asphalt_concrete': (0.25, 0.25, 0.25),
    'very_dry_ground': (0.80, 0.70, 0.50), 'medium_dry_ground': (0.60, 0.50, 0.35),
    'wet_ground': (0.40, 0.32, 0.22), 'clear_acrylic': (0.85, 0.95, 0.95), 'vacuum': (1, 1, 1),
}
SOLID_CATEGORIES = ('outer_wall', 'inner_wall', 'door')
OBJECT_CATEGORIES = ('key', 'ball', 'box')
TILE_CATEGORIES = ('goal', 'lava', 'floor')

In [ ]:
#| export
@dataclass
class SceneConfig:
    """Physical layout and materials of the exported scene."""
    cell_size: float = 2.0                  # metres per grid cell
    outer_wall_height: float = 3.0          # border walls
    inner_wall_height: float = 3.0          # interior walls / obstacles
    door_height: float | None = None        # closed doors (None: same as the wall height)
    object_height: float = 0.5              # keys, balls, boxes
    object_size: float = 0.4                # fraction of a cell
    antenna_height: float = 1.5             # height of agent / goal positions in scene.json
    center: bool = True                     # centre the grid at the world origin
    ground: bool = True
    ceiling: bool = False
    # category -> ITU material type (None: not exported)
    materials: dict = field(default_factory=lambda: {
        'outer_wall': 'concrete',
        'inner_wall': 'brick',
        'door': 'wood',
        'ground': 'concrete',
        'ceiling': 'ceiling_board',
        'key': 'metal',
        'ball': 'plywood',
        'box': 'chipboard',
        'goal': None,                       # non-physical markers are skipped by default
        'lava': None,
        'floor': None,
    })
    # category -> material thickness in metres (None: default, see `thickness_for`)
    thickness: dict = field(default_factory=dict)

    def thickness_for(self, category: str) -> float:
        if self.thickness.get(category) is not None:
            return float(self.thickness[category])
        if category in ('outer_wall', 'inner_wall'):
            return self.cell_size / 2      # two faces per cell -> cell_size in total
        if category == 'door':
            return 0.05
        if category in OBJECT_CATEGORIES:
            return 0.01
        return 0.1                          # ground, ceiling, tiles (Sionna's default)

## Materials

In [ ]:
#| export
def check_materials(config: SceneConfig, frequency: float | None = None):
    """Validate material names, and (optionally) that they are defined at `frequency` (Hz)."""
    problems = []
    for category, itu_type in config.materials.items():
        if itu_type is None:
            continue
        if itu_type not in ITU_MATERIALS:
            problems.append(f"{category}: unknown ITU material '{itu_type}' "
                            f"(available: {', '.join(ITU_MATERIALS)})")
        elif frequency is not None:
            f_ghz = frequency / 1e9
            if not any(lo <= f_ghz <= hi for lo, hi in ITU_MATERIALS[itu_type]):
                problems.append(f"{category}: '{itu_type}' is not defined at {f_ghz:g} GHz "
                                f"(valid ranges: {ITU_MATERIALS[itu_type]} GHz)")
    if problems:
        raise ValueError("invalid materials:\n  " + "\n  ".join(problems))

## Geometry

In [ ]:
#| export
class Mesh:
    """Triangle mesh built from quads, with outward-facing normals."""

    def __init__(self):
        self.vertices, self.faces = [], []

    def add_quad(self, corners, normal):
        corners = [np.asarray(c, dtype=float) for c in corners]
        n = np.cross(corners[1] - corners[0], corners[2] - corners[0])
        if np.dot(n, normal) < 0:          # make the winding match the outward normal
            corners = corners[::-1]
        base = len(self.vertices)
        self.vertices.extend(corners)
        self.faces.extend([(base, base + 1, base + 2), (base, base + 2, base + 3)])

    def add_box(self, center, size):
        (cx, cy, cz), (sx, sy, sz) = center, size
        x0, x1, y0, y1, z0, z1 = cx - sx / 2, cx + sx / 2, cy - sy / 2, cy + sy / 2, cz - sz / 2, cz + sz / 2
        self.add_quad([(x0, y0, z1), (x1, y0, z1), (x1, y1, z1), (x0, y1, z1)], (0, 0, 1))
        self.add_quad([(x0, y0, z0), (x1, y0, z0), (x1, y1, z0), (x0, y1, z0)], (0, 0, -1))
        self.add_quad([(x1, y0, z0), (x1, y1, z0), (x1, y1, z1), (x1, y0, z1)], (1, 0, 0))
        self.add_quad([(x0, y0, z0), (x0, y1, z0), (x0, y1, z1), (x0, y0, z1)], (-1, 0, 0))
        self.add_quad([(x0, y1, z0), (x1, y1, z0), (x1, y1, z1), (x0, y1, z1)], (0, 1, 0))
        self.add_quad([(x0, y0, z0), (x1, y0, z0), (x1, y0, z1), (x0, y0, z1)], (0, -1, 0))

    def add_sphere(self, center, radius, rings=6, segments=10):
        center = np.asarray(center, dtype=float)
        base = len(self.vertices)
        self.vertices.append(center + (0, 0, radius))
        for i in range(1, rings):
            theta = np.pi * i / rings
            for j in range(segments):
                phi = 2 * np.pi * j / segments
                self.vertices.append(center + radius * np.array(
                    [np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)]))
        self.vertices.append(center - (0, 0, radius))
        top, bottom = base, len(self.vertices) - 1
        ring = lambda i, j: base + 1 + (i - 1) * segments + j % segments
        for j in range(segments):
            self.faces.append((top, ring(1, j), ring(1, j + 1)))
            self.faces.append((bottom, ring(rings - 1, j + 1), ring(rings - 1, j)))
        for i in range(1, rings - 1):
            for j in range(segments):
                a, b, c, d = ring(i, j), ring(i, j + 1), ring(i + 1, j + 1), ring(i + 1, j)
                self.faces.extend([(a, d, c), (a, c, b)])

    def write_ply(self, path):
        vertices = np.asarray(self.vertices, dtype='<f4').reshape(-1, 3)
        faces = np.zeros(len(self.faces), dtype=[('n', 'u1'), ('idx', '<i4', (3,))])
        faces['n'], faces['idx'] = 3, np.asarray(self.faces, dtype='<i4').reshape(-1, 3)
        header = (f"ply\nformat binary_little_endian 1.0\nelement vertex {len(vertices)}\n"
                  "property float x\nproperty float y\nproperty float z\n"
                  f"element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n")
        with open(path, 'wb') as f:
            f.write(header.encode('ascii'))
            f.write(vertices.tobytes())
            f.write(faces.tobytes())

In [ ]:
#| export
def classify_cell(env, x: int, y: int) -> str | None:
    """Category of the object in grid cell (x, y), or None for an empty / open cell."""
    from multigrid.core.constants import Type
    obj = env.grid.get(x, y)
    if obj is None:
        return None
    if obj.type == Type.wall:
        border = x in (0, env.width - 1) or y in (0, env.height - 1)
        return 'outer_wall' if border else 'inner_wall'
    if obj.type == Type.door:
        return None if obj.is_open else 'door'
    if obj.type.value in OBJECT_CATEGORIES + TILE_CATEGORIES:
        return obj.type.value
    return None

In [ ]:
#| export
def grid_to_world(x: float, y: float, width: int, height: int, config: SceneConfig, z: float = 0.0):
    """World coordinates (metres) of the centre of grid cell (x, y)."""
    cs = config.cell_size
    wx, wy = (x + 0.5) * cs, -(y + 0.5) * cs
    if config.center:
        wx, wy = wx - width * cs / 2, wy + height * cs / 2
    return (float(wx), float(wy), float(z))

In [ ]:
#| export
def world_heading(direction: int) -> dict:
    """World heading of a grid direction: unit vector, and yaw (radians, counter-clockwise from +x)."""
    from multigrid.core.constants import Direction
    dx, dy = Direction(int(direction)).to_vec()
    heading = (float(dx), float(-dy), 0.0)      # grid y points down, world y points up
    return {'world_heading': heading, 'yaw': math.atan2(heading[1], heading[0])}

In [ ]:
#| export
def cell_layout(env, config: SceneConfig):
    """
    Per-cell layout of `env`, indexed [x][y]:
      * category of each cell (see `classify_cell`)
      * solid height of each cell (walls / closed doors, 0 elsewhere or if not exported)
      * surface height: height of the top surface in the cell (solid, object or tile), or 0
    """
    W, H = env.width, env.height
    categories = [[classify_cell(env, x, y) for y in range(H)] for x in range(W)]
    heights = {'outer_wall': config.outer_wall_height, 'inner_wall': config.inner_wall_height}

    def solid_height(x, y):
        category = categories[x][y]
        if category not in SOLID_CATEGORIES or config.materials.get(category) is None:
            return 0.0
        if category == 'door':
            if config.door_height is not None:
                return config.door_height
            # closed door: as high as the walls next to it
            return max([heights[c] for c in (categories[x + dx][y + dy]
                        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1))
                        if 0 <= x + dx < W and 0 <= y + dy < H) if c in heights]
                       or [config.outer_wall_height])
        return heights[category]

    solid = [[solid_height(x, y) for y in range(H)] for x in range(W)]
    size = config.object_size * config.cell_size
    object_tops = {'ball': size, 'box': size, 'key': 0.04, 'goal': 0.01, 'lava': 0.01, 'floor': 0.01}
    surface = [[solid[x][y] or (object_tops.get(categories[x][y], 0.0)
                                if config.materials.get(categories[x][y]) else 0.0)
                for y in range(H)] for x in range(W)]
    return categories, solid, surface

In [ ]:
#| export
def build_meshes(env, config: SceneConfig) -> dict[str, Mesh]:
    """Meshes (one per exported category) for the current layout of `env`."""
    W, H, cs = env.width, env.height, config.cell_size
    categories, solid, _ = cell_layout(env, config)

    def solid_height(x, y):
        return solid[x][y] if 0 <= x < W and 0 <= y < H else 0.0

    def cell_bounds(x, y):
        x0, y1, _ = grid_to_world(x - 0.5, y - 0.5, W, H, config)   # north-west corner
        return x0, x0 + cs, y1 - cs, y1                           # x0, x1, y0 (south), y1 (north)

    meshes: dict[str, Mesh] = {}
    mesh = lambda category: meshes.setdefault(category, Mesh())
    ceiling_height = (max(config.outer_wall_height, config.inner_wall_height)
                      if config.ceiling and config.materials.get('ceiling') else None)

    # Solids: top faces + side faces where the neighbour is lower (no internal faces)
    # grid neighbour -> (world outward normal)
    sides = {(1, 0): (1, 0, 0), (-1, 0): (-1, 0, 0), (0, -1): (0, 1, 0), (0, 1): (0, -1, 0)}
    for x in range(W):
        for y in range(H):
            h = solid_height(x, y)
            if h <= 0:
                continue
            m = mesh(categories[x][y])
            x0, x1, y0, y1 = cell_bounds(x, y)
            if not (ceiling_height is not None and abs(h - ceiling_height) < 1e-9):
                # (a top face in the ceiling plane would coincide with the ceiling: skip it)
                m.add_quad([(x0, y0, h), (x1, y0, h), (x1, y1, h), (x0, y1, h)], (0, 0, 1))
            for (dx, dy), normal in sides.items():
                h_n = solid_height(x + dx, y + dy)
                if h_n >= h:
                    continue
                if normal == (1, 0, 0):
                    edge = [(x1, y0), (x1, y1)]
                elif normal == (-1, 0, 0):
                    edge = [(x0, y0), (x0, y1)]
                elif normal == (0, 1, 0):
                    edge = [(x0, y1), (x1, y1)]
                else:
                    edge = [(x0, y0), (x1, y0)]
                (ax, ay), (bx, by) = edge
                m.add_quad([(ax, ay, h_n), (bx, by, h_n), (bx, by, h), (ax, ay, h)], normal)

    # Small objects and flat tiles
    size = config.object_size * cs
    for x in range(W):
        for y in range(H):
            category = categories[x][y]
            if config.materials.get(category) is None:
                continue
            cx, cy, _ = grid_to_world(x, y, W, H, config)
            if category == 'ball':
                mesh(category).add_sphere((cx, cy, size / 2), size / 2)
            elif category == 'box':
                mesh(category).add_box((cx, cy, size / 2), (size, size, size))
            elif category == 'key':
                mesh(category).add_box((cx, cy, 0.02), (size, size / 3, 0.04))
            elif category in TILE_CATEGORIES:
                x0, x1, y0, y1 = cell_bounds(x, y)
                mesh(category).add_quad([(x0, y0, 0.01), (x1, y0, 0.01), (x1, y1, 0.01), (x0, y1, 0.01)], (0, 0, 1))

    # Ground and ceiling over the whole grid
    gx0, _, _, gy1 = cell_bounds(0, 0)
    _, gx1, gy0, _ = cell_bounds(W - 1, H - 1)
    if config.ground and config.materials.get('ground'):
        mesh('ground').add_quad([(gx0, gy0, 0), (gx1, gy0, 0), (gx1, gy1, 0), (gx0, gy1, 0)], (0, 0, 1))
    if config.ceiling and config.materials.get('ceiling'):
        hc = max(config.outer_wall_height, config.inner_wall_height)
        mesh('ceiling').add_quad([(gx0, gy0, hc), (gx1, gy0, hc), (gx1, gy1, hc), (gx0, gy1, hc)], (0, 0, -1))

    return meshes

## Export

In [ ]:
#| export
def material_ids(config: SceneConfig, categories) -> dict[str, tuple[str, str, float]]:
    """category -> (material id, ITU type, thickness); one material per (type, thickness)."""
    groups = {}
    for category in categories:
        key = (config.materials[category], config.thickness_for(category))
        groups.setdefault(key, []).append(category)
    types = [itu_type for itu_type, _ in groups]
    out = {}
    for (itu_type, thickness), cats in groups.items():
        mat_id = f"mat-itu_{itu_type}"
        if types.count(itu_type) > 1:      # same type with different thicknesses
            mat_id += f"-t{thickness:g}".replace('.', 'p')
        for category in cats:
            out[category] = (mat_id, itu_type, thickness)
    return out

In [ ]:
#| export
def write_sionna_xml(path, meshes, mats):
    lines = ['<scene version="2.1.0">',
             '    <!-- Generated by multigrid.utils.sionna_export: ITU radio materials (Sionna RT) -->']
    written = set()
    for category in meshes:
        mat_id, itu_type, thickness = mats[category]
        if mat_id in written:
            continue
        written.add(mat_id)
        r, g, b = ITU_COLORS.get(itu_type, (0.5, 0.5, 0.5))
        lines += [f'    <bsdf type="itu-radio-material" id="{mat_id}">',
                  f'        <string name="type" value="{itu_type}"/>',
                  f'        <float name="thickness" value="{thickness:g}"/>',
                  f'        <rgb name="color" value="{r:.3f} {g:.3f} {b:.3f}"/>',
                  '    </bsdf>']
    for category in meshes:
        lines += [f'    <shape type="ply" id="mesh-{category}">',
                  f'        <string name="filename" value="meshes/{category}.ply"/>',
                  f'        <ref id="{mats[category][0]}" name="bsdf"/>',
                  '    </shape>']
    lines.append('</scene>')
    with open(path, 'w') as f:
        f.write('\n'.join(lines) + '\n')

In [ ]:
#| export
def default_out_dir(name: str, seed: int, kwargs: dict, root: str = 'sionna_scenes') -> str:
    """
    Unique folder for a scene: `<root>/<name>_seed<seed>`, plus a short hash of the env
    kwargs when there are any (the exact kwargs are stored in scene.json).
    """
    folder = f"{name}_seed{seed}"
    if kwargs:
        digest = hashlib.sha1(json.dumps(kwargs, sort_keys=True, default=str).encode()).hexdigest()[:6]
        folder += f"_{digest}"
    return os.path.join(root, folder)

In [ ]:
#| export
def clear_scene_dir(out_dir: str):
    """Remove the files of a previous export (only the files this script writes)."""
    for name in ('scene.xml', 'scene.json', 'scene.blend', 'scene.blend1'):
        path = os.path.join(out_dir, name)
        if os.path.exists(path):
            os.remove(path)
    mesh_dir = os.path.join(out_dir, 'meshes')
    if os.path.isdir(mesh_dir):
        for name in os.listdir(mesh_dir):
            if name.endswith('.ply'):
                os.remove(os.path.join(mesh_dir, name))

In [ ]:
#| export
def export_scene(env, out_dir: str, config: SceneConfig | None = None, frequency: float | None = None,
                 blend: bool = False, blender: str | None = None, info: dict | None = None,
                 overwrite: bool = False) -> dict:
    """
    Export the current layout of `env` (any MultiGrid env, already reset) to a Sionna RT scene.

    Raises `FileExistsError` if `out_dir` already holds a scene, unless `overwrite=True`
    (which first removes the previous scene's files, so no stale meshes are left).
    Returns the scene metadata (also written to `<out_dir>/scene.json`).
    """
    config = config or SceneConfig()
    check_materials(config, frequency)
    env = env.unwrapped
    if os.path.exists(os.path.join(out_dir, 'scene.xml')):
        if not overwrite:
            raise FileExistsError(
                f"{os.path.abspath(out_dir)} already contains a scene "
                "(use another output folder, or overwrite=True / --overwrite to replace it)")
        clear_scene_dir(out_dir)
    os.makedirs(os.path.join(out_dir, 'meshes'), exist_ok=True)

    meshes = build_meshes(env, config)
    mats = material_ids(config, meshes)
    for category, mesh in meshes.items():
        mesh.write_ply(os.path.join(out_dir, 'meshes', f'{category}.ply'))
    write_sionna_xml(os.path.join(out_dir, 'scene.xml'), meshes, mats)

    from multigrid.core.constants import Type
    W, H = env.width, env.height
    world = lambda x, y, z=config.antenna_height: grid_to_world(int(x), int(y), W, H, config, z)
    goals = [(x, y) for x in range(W) for y in range(H)
             if (o := env.grid.get(x, y)) is not None and o.type == Type.goal]
    meta = {
        'source': {'env': type(env).__name__, **(info or {})},
        'grid': {'width': W, 'height': H},
        'config': asdict(config),
        'mapping': {
            'cell_size': config.cell_size,
            'origin': list(grid_to_world(-0.5, -0.5, W, H, config)),   # world position of the grid's top-left corner
            'formula': 'world = origin + cell_size * (x + 0.5, -(y + 0.5), 0); z up, metres',
        },
        'materials': {category: {'id': mat_id, 'itu_type': itu_type, 'thickness': thickness,
                                 'valid_ghz': ITU_MATERIALS[itu_type]}
                      for category, (mat_id, itu_type, thickness) in mats.items()},
        'meshes': {category: {'file': f'meshes/{category}.ply', 'triangles': len(mesh.faces)}
                   for category, mesh in meshes.items()},
        'agents': [{'index': a.index, 'color': a.state.color.value,
                    'grid_pos': [int(p) for p in a.state.pos], 'dir': int(a.state.dir),
                    'world_pos': world(*a.state.pos), **world_heading(a.state.dir)}
                   for a in env.agents if not a.state.terminated],
        'goals': [{'grid_pos': [x, y], 'world_pos': world(x, y)} for x, y in goals],
        # per-cell layout, indexed [x][y] (used by check_sionna_scene.py)
        'layout': dict(zip(('categories', 'solid_height', 'surface_height'), cell_layout(env, config))),
    }
    with open(os.path.join(out_dir, 'scene.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    if blend:
        build_blend_subprocess(out_dir, blender)
    return meta

## Blender

In [ ]:
#| export
def build_blend(out_dir: str):
    """Build `<out_dir>/scene.blend` from the exported meshes (runs where `bpy` is importable)."""
    import bpy
    out_dir = os.path.abspath(out_dir)
    with open(os.path.join(out_dir, 'scene.json')) as f:
        meta = json.load(f)

    bpy.ops.wm.read_factory_settings(use_empty=True)
    materials = {}
    for category, info in meta['meshes'].items():
        bpy.ops.wm.ply_import(filepath=os.path.join(out_dir, info['file']))
        obj = bpy.context.selected_objects[0]
        obj.name = obj.data.name = category

        mat_info = meta['materials'][category]
        name = f"itu_{mat_info['itu_type']}"       # exported by mitsuba-blender as mat-itu_<type>
        if name not in materials:
            mat = bpy.data.materials.new(name)
            mat.use_nodes = True
            bsdf = mat.node_tree.nodes.get('Principled BSDF')
            color = ITU_COLORS.get(mat_info['itu_type'], (0.5, 0.5, 0.5))
            if bsdf is not None:
                bsdf.inputs['Base Color'].default_value = (*color, 1.0)
            mat.diffuse_color = (*color, 1.0)
            mat['itu_type'], mat['thickness'] = mat_info['itu_type'], mat_info['thickness']
            materials[name] = mat
        obj.data.materials.clear()
        obj.data.materials.append(materials[name])

    # Agents and goals as empties (not exported as geometry), e.g. to place transmitters / receivers
    for agent in meta['agents']:
        empty = bpy.data.objects.new(f"agent_{agent['index']}", None)
        empty.location, empty.empty_display_type = agent['world_pos'], 'SINGLE_ARROW'
        bpy.context.scene.collection.objects.link(empty)
    for k, goal in enumerate(meta['goals']):
        empty = bpy.data.objects.new(f"goal_{k}", None)
        empty.location, empty.empty_display_type = goal['world_pos'], 'SPHERE'
        bpy.context.scene.collection.objects.link(empty)

    bpy.ops.wm.save_as_mainfile(filepath=os.path.join(out_dir, 'scene.blend'))
    print(f"saved {os.path.join(out_dir, 'scene.blend')} with Blender {bpy.app.version_string}")

In [ ]:
#| export
def _module_file() -> str:
    """
    Source file of this module. It is run as a plain script to build the .blend (also inside a
    Blender binary, whose Python may not have multigrid installed), so it only imports the
    standard library and numpy at module level.
    """
    if '__file__' in globals():
        return os.path.abspath(__file__)
    import importlib.util                          # e.g. when called from the notebook
    return importlib.util.find_spec('multigrid.utils.sionna_export').origin

In [ ]:
#| export
def build_blend_subprocess(out_dir: str, blender: str | None = None):
    """
    Build the .blend in a separate process, with the Blender binary `blender` (or the
    BLENDER environment variable), else with the `bpy` module of this Python.

    A .blend file can only be opened by the Blender version that saved it, or newer
    (e.g. a file saved by Blender 5.x cannot be opened in Blender 4.2), so pass the
    Blender you will open the scene with.
    """
    script = _module_file()
    blender = blender or os.environ.get('BLENDER')
    if blender:
        cmd = [blender, '-b', '--factory-startup', '-P', script, '--', '--build-blend', out_dir]
    else:
        cmd = [sys.executable, script, '--build-blend', out_dir]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0 or not os.path.exists(os.path.join(out_dir, 'scene.blend')):
        raise RuntimeError(
            "building the .blend failed (install the `bpy` module or pass --blender /path/to/blender)\n"
            + result.stdout[-2000:] + result.stderr[-2000:])
    saved = [line for line in result.stdout.splitlines() if line.startswith('saved ')]
    print(saved[-1] if saved else f"saved {os.path.join(out_dir, 'scene.blend')}")
    if not blender:
        print("note: built with the `bpy` module; the .blend opens only in that Blender version or newer "
              "(use --blender /path/to/blender or set BLENDER to build it for your Blender)")

## Command line

In [ ]:
#| export
def parse_kwargs(pairs):
    out = {}
    for pair in pairs or []:
        key, value = pair.split('=', 1)
        for cast in (int, float):
            try:
                value = cast(value)
                break
            except ValueError:
                pass
        else:
            value = {'true': True, 'false': False, 'none': None}.get(value.lower(), value)
        out[key] = value
    return out

In [ ]:
#| export
def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--build-blend', metavar='SCENE_DIR',
                        help="only build SCENE_DIR/scene.blend from an already exported scene, then exit")
    parser.add_argument('--env', default='MultiGrid-FindGoal-15x15-v0', help="registered env id")
    parser.add_argument('--config', help="Hydra/YAML env config (e.g. multigrid/configs/env/findgoal.yaml), instead of --env")
    parser.add_argument('--env-kwargs', nargs='*', default=[], help="extra env kwargs as key=value")
    parser.add_argument('--agents', type=int, default=None)
    parser.add_argument('--seed', type=int, default=0)
    parser.add_argument('--out', default=None,
                        help="output folder (default: sionna_scenes/<env>_seed<seed>[_<kwargs hash>])")
    parser.add_argument('--overwrite', action='store_true', help="replace an existing scene in the output folder")
    parser.add_argument('--cell-size', type=float, default=2.0)
    parser.add_argument('--wall-height', type=float, default=3.0, help="outer wall height")
    parser.add_argument('--inner-wall-height', type=float, default=None, help="default: --wall-height")
    parser.add_argument('--door-height', type=float, default=None)
    parser.add_argument('--antenna-height', type=float, default=1.5)
    parser.add_argument('--material', nargs='*', default=[], metavar='CATEGORY=ITU_TYPE',
                        help="e.g. inner_wall=metal goal=metal box=none")
    parser.add_argument('--thickness', nargs='*', default=[], metavar='CATEGORY=METRES')
    parser.add_argument('--ceiling', metavar='ITU_TYPE', default=None, help="add a ceiling (indoor)")
    parser.add_argument('--no-ground', action='store_true')
    parser.add_argument('--no-objects', action='store_true', help="skip keys, balls and boxes")
    parser.add_argument('--no-center', action='store_true')
    parser.add_argument('--frequency', type=float, default=None, help="carrier frequency in Hz to validate materials")
    parser.add_argument('--blend', action='store_true', help="also build scene.blend")
    parser.add_argument('--blender', default=None,
                        help="Blender binary used to build scene.blend (default: $BLENDER, else the bpy module). "
                             "Use the Blender you will open the file with: older Blenders cannot open newer .blend files")
    parser.add_argument('--list-materials', action='store_true')
    args = parser.parse_args(argv)

    if args.build_blend:
        build_blend(args.build_blend)
        return
    if args.list_materials:
        for name, ranges in ITU_MATERIALS.items():
            print(f"  {name:18s} {', '.join(f'{lo:g}-{hi:g}' for lo, hi in ranges)} GHz")
        return

    config = SceneConfig(
        cell_size=args.cell_size, outer_wall_height=args.wall_height,
        inner_wall_height=args.inner_wall_height or args.wall_height, door_height=args.door_height,
        antenna_height=args.antenna_height, center=not args.no_center,
        ground=not args.no_ground, ceiling=args.ceiling is not None,
    )
    if args.ceiling:
        config.materials['ceiling'] = args.ceiling
    if args.no_objects:
        for category in OBJECT_CATEGORIES:
            config.materials[category] = None
    for category, value in parse_kwargs(args.material).items():
        if category not in config.materials:
            parser.error(f"unknown category '{category}' (categories: {', '.join(config.materials)})")
        config.materials[category] = value
    config.thickness.update({k: float(v) for k, v in parse_kwargs(args.thickness).items()})

    # Create and reset the env
    import gymnasium as gym
    import multigrid.envs  # noqa: F401  (registers the envs)
    kwargs = parse_kwargs(args.env_kwargs)
    if args.agents is not None:
        kwargs['agents'] = args.agents
    with contextlib.redirect_stdout(io.StringIO()):   # some envs print debug info
        if args.config:
            from hydra.utils import instantiate
            from omegaconf import OmegaConf
            cfg = OmegaConf.merge(OmegaConf.load(args.config), kwargs)
            env, source = instantiate(cfg), {'config': args.config}
        else:
            env, source = gym.make(args.env, **kwargs), {'env_id': args.env}
        env.reset(seed=args.seed)
    source.update({'seed': args.seed, 'env_kwargs': kwargs})

    if args.out is None:
        name = (os.path.splitext(os.path.basename(args.config))[0] if args.config
                else args.env.removeprefix('MultiGrid-').removesuffix('-v0'))
        args.out = default_out_dir(name, args.seed, kwargs)
    try:
        meta = export_scene(env, args.out, config, frequency=args.frequency, blend=args.blend,
                            blender=args.blender, info=source, overwrite=args.overwrite)
    except FileExistsError as e:
        parser.exit(1, f"error: {e}\n")
    print(f"exported {meta['source']['env']} ({meta['grid']['width']}x{meta['grid']['height']}) "
          f"to {os.path.abspath(args.out)}")
    for category, info in meta['materials'].items():
        print(f"  {category:11s} {info['id']:24s} thickness={info['thickness']:g} m "
              f"({meta['meshes'][category]['triangles']} triangles)")
    print(f"  agents: {[a['world_pos'] for a in meta['agents']]}")

In [ ]:
#| export
if __name__ == '__main__' and 'ipykernel' not in sys.modules:   # not when running the notebook
    # Inside Blender (`blender -b -P this_file -- ...`), arguments come after '--'
    main(sys.argv[sys.argv.index('--') + 1:] if '--' in sys.argv else None)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()